# GrowthParameterEstimation Function Tour

This notebook is a self-contained tour of the package API. It manufactures a small dataset, exercises the data, exposure, model, registry, simulation, observation, fitting, analysis, and workflow functions, then runs a staged fitting example:

1. untreated monocultures
2. treated monocultures
3. untreated cocultures
4. treated cocultures

All tables and run artifacts are written under `tests/outputs/function_tour`.

In [ ]:
using Pkg
project_root = normpath(joinpath(@__DIR__, "..", ".."))
Pkg.activate(project_root)

try
    @eval using GrowthParameterEstimation
catch err
    @warn "Package load failed; instantiating project before retrying" exception=(err, catch_backtrace())
    Pkg.instantiate()
    @eval using GrowthParameterEstimation
end

using CSV, DataFrames, Dates, Random, Statistics
using OrdinaryDiffEq, DifferentialEquations

try
    @eval using Plots
catch err
    @warn "Plots is unavailable; plot-generating cells will still export CSV artifacts" exception=(err, catch_backtrace())
end

Random.seed!(42)
output_dir = joinpath(@__DIR__, "..", "outputs", "function_tour")
mkpath(output_dir)
println("Project root: ", project_root)
println("Output dir: ", output_dir)

## Public API Inventory

The first table lists every exported package symbol so the notebook remains a true function tour as the package evolves.

In [ ]:
api_symbols = sort(String.(names(GrowthParameterEstimation; all=false)))
api_inventory = DataFrame(symbol=api_symbols)
CSV.write(joinpath(output_dir, "api_inventory.csv"), api_inventory)
println("Exported public symbols: ", nrow(api_inventory))
display(api_inventory)

## Manufactured Multistage Dataset

The data are now generated by concrete registered model specifications: logistic and Gompertz monocultures, theta-logistic drug death monocultures, cooperative coculture, and cooperative coculture with population-specific drug death terms.


In [ ]:
times = collect(0.0:1.0:8.0)
rows = NamedTuple[]

function simulate_registered(model_name, params, u0; dose=0.0)
    spec = get_model(model_name)
    sim = simulate(spec, times, Float64.(params); u0=Float64.(u0), exposure=ConstantExposure(Float64(dose)))
    sim.success || error("Synthetic generator failed for " * model_name * ": " * sim.reason)
    return sim
end

function add_simulated_condition!(rows, sim; culture_type, population_type, dose, treatment_amount, density, replicate,
                                  initial_sensitive, initial_resistant, source_model, true_parameters)
    for (i, t) in enumerate(times)
        expected_total = Float64(sim.observed[i])
        observed_total = max(1e-6, expected_total * (1 + 0.018 * randn()))
        sensitive_state = size(sim.states, 1) >= 1 ? Float64(sim.states[1, i]) : expected_total
        resistant_state = size(sim.states, 1) >= 2 ? Float64(sim.states[2, i]) : 0.0
        push!(rows, (
            time=Float64(t),
            count=observed_total,
            error=max(0.045 * observed_total, 0.5),
            dose=Float64(dose),
            cell_line="A549",
            density=Float64(density),
            replicate=replicate,
            culture_type=culture_type,
            population_type=population_type,
            treatment_amount=Float64(treatment_amount),
            initial_sensitive=Float64(initial_sensitive),
            initial_resistant=Float64(initial_resistant),
            model_generator=source_model,
            true_parameters=join(round.(Float64.(true_parameters), digits=4), ";"),
            latent_sensitive=sensitive_state,
            latent_resistant=resistant_state,
            unit_time="h",
            unit_count="cells",
        ))
    end
    return rows
end

# Stage 1: untreated monocultures from simple baseline growth laws.
naive_logistic_params = [0.48, 1350.0]
resistant_gompertz_params = [0.24, 1.0, 1125.0]
naive_control = simulate_registered("logistic_growth", naive_logistic_params, [120.0])
resistant_control = simulate_registered("gompertz_growth", resistant_gompertz_params, [90.0])
add_simulated_condition!(rows, naive_control; culture_type="monoculture", population_type="naive", dose=0.0,
    treatment_amount=0.0, density=120.0, replicate=1, initial_sensitive=120.0, initial_resistant=0.0,
    source_model="logistic_growth", true_parameters=naive_logistic_params)
add_simulated_condition!(rows, resistant_control; culture_type="monoculture", population_type="resistant", dose=0.0,
    treatment_amount=0.0, density=90.0, replicate=1, initial_sensitive=0.0, initial_resistant=90.0,
    source_model="gompertz_growth", true_parameters=resistant_gompertz_params)

# Stage 2: treated monocultures from multimodifier theta-logistic drug death models.
naive_kill_params = [0.48, 1350.0, 1.15, 0.42, 0.32, 1.45]
resistant_kill_params = [0.36, 1125.0, 1.05, 0.22, 1.25, 1.25]
for dose in (0.25, 0.75)
    naive_treated = simulate_registered("theta_logistic_hill_kill", naive_kill_params, [120.0]; dose=dose)
    resistant_treated = simulate_registered("theta_logistic_hill_kill", resistant_kill_params, [90.0]; dose=dose)
    add_simulated_condition!(rows, naive_treated; culture_type="monoculture", population_type="naive", dose=dose,
        treatment_amount=dose, density=120.0, replicate=1, initial_sensitive=120.0, initial_resistant=0.0,
        source_model="theta_logistic_hill_kill", true_parameters=naive_kill_params)
    add_simulated_condition!(rows, resistant_treated; culture_type="monoculture", population_type="resistant", dose=dose,
        treatment_amount=dose, density=90.0, replicate=1, initial_sensitive=0.0, initial_resistant=90.0,
        source_model="theta_logistic_hill_kill", true_parameters=resistant_kill_params)
end

# Stage 3: untreated coculture from an explicit cooperative Lotka-Volterra system.
coop_params = [0.42, 1300.0, 0.24, 0.31, 1050.0, 0.18]
coop_control = simulate_registered("lotka_volterra_cooperation", coop_params, [120.0, 90.0])
add_simulated_condition!(rows, coop_control; culture_type="coculture", population_type="mixed", dose=0.0,
    treatment_amount=0.0, density=210.0, replicate=1, initial_sensitive=120.0, initial_resistant=90.0,
    source_model="lotka_volterra_cooperation", true_parameters=coop_params)

# Stage 4: treated coculture from cooperation plus population-specific drug death terms.
coop_drug_params = [0.42, 1300.0, 0.24, 0.31, 1050.0, 0.18, 0.48, 0.34, 0.20, 1.30, 1.35]
for dose in (0.25, 0.75)
    coop_treated = simulate_registered("lotka_volterra_hill_cooperation", coop_drug_params, [120.0, 90.0]; dose=dose)
    add_simulated_condition!(rows, coop_treated; culture_type="coculture", population_type="mixed", dose=dose,
        treatment_amount=dose, density=210.0, replicate=1, initial_sensitive=120.0, initial_resistant=90.0,
        source_model="lotka_volterra_hill_cooperation", true_parameters=coop_drug_params)
end

manufactured = DataFrame(rows)
sort!(manufactured, [:time, :culture_type, :population_type, :dose])
CSV.write(joinpath(output_dir, "manufactured_multistage_data.csv"), manufactured)
model_story = combine(groupby(manufactured, [:culture_type, :population_type, :dose, :model_generator]), nrow => :rows)
CSV.write(joinpath(output_dir, "manufactured_model_story.csv"), model_story)
println("Manufactured rows: ", nrow(manufactured))
display(model_story)
display(first(manufactured, 12))

## Data And Exposure Functions

This cell covers `normalize_schema`, `validate_timeseries`, `validate_required_metadata`, `load_timeseries`, `build_exposure`, and `evaluate_exposure`.

In [ ]:
loaded = load_timeseries(joinpath(output_dir, "manufactured_multistage_data.csv"))
normalized = normalize_schema(loaded)
@assert validate_timeseries(normalized)
@assert validate_required_metadata(normalized)

constant_exposure = build_exposure(:constant; value=0.25)
pulse_exposure = build_exposure(:pulse; amplitude=0.75, start_time=2.0, end_time=5.0)
stepped_exposure = build_exposure(:stepped; change_times=[0.0, 3.0, 6.0], values=[0.0, 0.25, 0.75])
decaying_exposure = build_exposure(:decay; c0=1.0, decay_rate=0.35, t0=0.0)

exposure_table = DataFrame(
    time=times,
    constant=evaluate_exposure(constant_exposure, times),
    pulse=evaluate_exposure(pulse_exposure, times),
    stepped=evaluate_exposure(stepped_exposure, times),
    decaying=evaluate_exposure(decaying_exposure, times),
)
CSV.write(joinpath(output_dir, "exposure_profiles.csv"), exposure_table)
display(exposure_table)

## Models, Registry, Simulation, Observation, And Sweeps

This cell covers composable model builders/modifiers, the model registry, simulation, observation helpers, and parameter sweeps.

In [ ]:
base_logistic = GrowthParameterEstimation.Models.build_logistic(r=0.45, K=1200.0)
base_gompertz = GrowthParameterEstimation.Models.build_gompertz(a=0.20, b=1.0, K=1200.0)
base_exponential = GrowthParameterEstimation.Models.build_exponential(r=0.25)
logistic_with_death = GrowthParameterEstimation.Models.apply_death(base_logistic; death_rate=0.03)
logistic_with_lag = GrowthParameterEstimation.Models.apply_lag(base_logistic; tlag=1.5)
logistic_with_inhibition = GrowthParameterEstimation.Models.apply_hill_inhibition(base_logistic; emax=0.8, ic50=0.4, hill=1.3)
logistic_with_kill = GrowthParameterEstimation.Models.apply_hill_kill(base_logistic; emax_kill=0.25, ic50=0.4, hill=1.3)
composed_model = GrowthParameterEstimation.Models.compose_models(base_logistic, [GrowthParameterEstimation.Models.DeathModifier]; death_rate=0.02)

model_examples = DataFrame(
    model=["logistic", "gompertz", "exponential", "death", "lag", "hill_inhibition", "hill_kill", "composed_death"],
    derivative_at_200=[
        base_logistic(200.0, (), 2.0),
        base_gompertz(200.0, (), 2.0),
        base_exponential(200.0, (), 2.0),
        logistic_with_death(200.0, (), 2.0),
        logistic_with_lag(200.0, (), 1.0),
        logistic_with_inhibition(200.0, Dict(:drug => 0.5), 2.0),
        logistic_with_kill(200.0, Dict(:drug => 0.5), 2.0),
        composed_model(200.0, (), 2.0),
    ],
)

registered_models = list_models()
logistic_spec = get_model("logistic_growth")
families = DataFrame(family=["logistic", "gompertz", "coculture", "mechanistic"], n_models=[length(models_by_family(f)) for f in ["logistic", "gompertz", "coculture", "mechanistic"]])

sim = simulate(logistic_spec, times, [0.48, 1350.0]; u0=[120.0], exposure=ConstantExposure(0.0))
obs_spec = ObservationSpec("scaled_viable", viable_total, 1.1, 5.0)
sum_first_two = sum_states([1, 2])
obs_table = DataFrame(raw_total=[sum_first_two([100.0, 25.0], nothing, 0.0), viable_total([100.0, 25.0], nothing, 0.0)], observed_signal=[observed_signal(obs_spec, [100.0, 25.0], nothing, 0.0), observed_signal(obs_spec, [200.0], nothing, 0.0)])
CSV.write(joinpath(output_dir, "observation_examples.csv"), obs_table)

grid = SweepGrid([120.0, 210.0], [0.0, 0.25], [0.0, 0.75], times)
sweep = run_sweep(logistic_spec, [0.48, 1350.0], grid)

CSV.write(joinpath(output_dir, "model_examples.csv"), model_examples)
CSV.write(joinpath(output_dir, "model_families.csv"), families)
CSV.write(joinpath(output_dir, "sweep_summary.csv"), sweep.summary)

println("Registered models: ", join(registered_models, ", "))
println("Logistic simulation success: ", sim.success, " final observed=", round(sim.observed[end], digits=2))
display(model_examples)
display(families)
display(obs_table)
display(first(sweep.summary, 8))

## Single-Condition Fitting And Analysis

This cell covers `setUpProblem`, `calculate_bic`, `pQuickStat`, `run_single_fit`, model comparisons, validation, sensitivity, residual, and enhanced BIC helpers.

In [ ]:
mono_naive = subset(normalized, :culture_type => ByRow(==("monoculture")), :population_type => ByRow(==("naive")), :dose => ByRow(==(0.0)))
x = Float64.(mono_naive.time)
y = Float64.(mono_naive.count)
p0 = [0.3, maximum(y) * 1.8]
bounds = [(0.01, 2.0), (maximum(y), 5000.0)]
logistic_ode! = GrowthParameterEstimation.Models.to_ode!(GrowthParameterEstimation.Models.build_logistic())
gompertz_ode! = GrowthParameterEstimation.Models.to_ode!(GrowthParameterEstimation.Models.build_gompertz())

p_opt, sol_opt, prob_opt = setUpProblem(logistic_ode!, x, y, Tsit5(), [y[1]], p0, (minimum(x), maximum(x)), bounds; maxiters=400)
bic, ssr = calculate_bic(prob_opt, x, y, Tsit5(), p_opt)
pQuickStat(x, y, p_opt, sol_opt, prob_opt, bic, ssr)

single_fit = run_single_fit(x, y, p0; model=logistic_ode!, solver=Tsit5(), bounds=bounds, show_stats=false)
comparison = compare_models(x, y, "Logistic", logistic_ode!, p0, "Gompertz", gompertz_ode!, [0.2, 1.0, maximum(y) * 1.8]; solver=Tsit5(), bounds1=bounds, bounds2=[(0.01, 2.0), (0.1, 5.0), (maximum(y), 5000.0)], output_csv=joinpath(output_dir, "single_compare_models.csv"))
model_specs = Dict(
    "Logistic" => (model=logistic_ode!, p0=p0, bounds=bounds),
    "Gompertz" => (model=gompertz_ode!, p0=[0.2, 1.0, maximum(y) * 1.8], bounds=[(0.01, 2.0), (0.1, 5.0), (maximum(y), 5000.0)]),
)
all_fits = compare_models_dict(x, y, model_specs; default_solver=Tsit5(), output_csv=joinpath(output_dir, "single_compare_models_dict.csv"))

loo = leave_one_out_validation(x, y, p0; model=logistic_ode!, solver=Tsit5(), bounds=bounds, show_stats=false)
kfold = k_fold_cross_validation(x, y, p0; k_folds=3, model=logistic_ode!, solver=Tsit5(), bounds=bounds, show_stats=false)
sensitivity = parameter_sensitivity_analysis(x, y, single_fit; perturbation=0.1, model=logistic_ode!, solver=Tsit5())
residuals = residual_analysis(x, y, single_fit; model=logistic_ode!, solver=Tsit5())
enhanced = enhanced_bic_analysis(x, y; models=[logistic_ode!, gompertz_ode!], model_names=["Logistic", "Gompertz"], p0_values=[p0, [0.2, 1.0, maximum(y) * 1.8]], solver=Tsit5())

# Unified registered-model fitting helpers
registered_fit = fit_model(logistic_spec, x, y, 0.0; maxiters=250, reltol=1e-6, abstol=1e-6)
condition_df = DataFrame(time=x, count=y, dose=fill(0.0, length(x)), condition=fill("naive_control", length(x)))
condition_fit = fit_condition(condition_df, "naive_control", [logistic_spec]; untreated_baseline=(r=registered_fit.params[1], K=registered_fit.params[2]))

# Joint fitting helpers on a tiny two-state manufactured example
function joint_exp!(du, u, p, t)
    du[1] = p[1] * u[1]
    du[2] = p[2] * u[2]
    return nothing
end
joint_dataset_specs = [
    (x=x, y=120.0 .* exp.(0.18 .* x), state_index=1),
    (x=x, y=90.0 .* exp.(0.10 .* x), state_index=2),
]
joint_fit = run_joint_fit(joint_exp!, joint_dataset_specs, [120.0, 90.0], [0.15, 0.08]; solver=Tsit5(), bounds=[(0.001, 1.0), (0.001, 1.0)], maxiters=300)
joint_fits = compare_joint_models_dict(
    joint_dataset_specs,
    [120.0, 90.0],
    Dict("joint_exponential" => (model=joint_exp!, p0=[0.15, 0.08], bounds=[(0.001, 1.0), (0.001, 1.0)]));
    default_solver=Tsit5(),
    output_csv=joinpath(output_dir, "joint_model_comparison.csv"),
)

fit_summary = DataFrame(
    metric=["setUpProblem_r", "setUpProblem_K", "single_fit_bic", "single_fit_ssr", "best_comparison", "loo_rmse", "kfold_rmse", "enhanced_best", "fit_model_bic", "fit_condition_rows", "joint_fit_bic"],
    value=string.([p_opt[1], p_opt[2], single_fit.bic, single_fit.ssr, comparison.best_model.name, loo.rmse, kfold.overall_rmse, enhanced.best_model.model_name, registered_fit.bic, nrow(condition_fit), joint_fit.bic]),
)
CSV.write(joinpath(output_dir, "basic_fit_and_analysis_summary.csv"), fit_summary)
display(fit_summary)

## Workflow And Staged Pipeline

This cell covers `FitCondition`, `PipelineConfig`, `PipelineStage`, config I/O, condition building, ranking, plotting, exporting, and the staged pipeline. The staged pipeline uses `default_population_cellline_stages` so it explicitly moves from monocultures to treated monocultures, untreated coculture, and treated coculture.

In [ ]:
conditions = build_conditions(normalized; condition_cols=[:culture_type, :population_type, :dose, :cell_line, :density, :replicate])
println("Workflow conditions: ", length(conditions))

cfg = default_config(output_dir=output_dir)
cfg = PipelineConfig(cfg.version, cfg.model_names, 3, 2, 160, cfg.reltol, cfg.abstol, cfg.weighted, 42, cfg.output_dir)
cfg_path = save_config(joinpath(output_dir, "function_tour_config.toml"), cfg)
loaded_cfg = load_config(cfg_path)

qc_report = generate_qc_report(normalized)
qc_paths = save_qc_report(qc_report; output_dir=joinpath(output_dir, "diagnostics"))
preflight = preflight_data_quality(normalized; stages=default_population_cellline_stages(normalized))
preflight_paths = save_preflight_report(preflight; output_dir=joinpath(output_dir, "diagnostics"))

quick_rank = rank_models(["logistic_growth", "gompertz_growth"], conditions[1:1]; n_starts=3, maxiters=100, top_k=2, seed=42)
plot_paths = plot_topk(quick_rank; conditions=conditions[1:1], top_k=2, output_dir=joinpath(output_dir, "quick_figures"))
quick_exports = export_results(quick_rank; output_dir=joinpath(output_dir, "quick_rank"))

stages = default_population_cellline_stages(normalized; populations=["naive", "resistant"])
staged = run_staged_pipeline(
    normalized;
    stages=stages,
    config=loaded_cfg,
    selection_mode=:best_bic,
    export_stage_results=true,
    qc_before_fit=true,
    preflight_before_fit=true,
    n_bootstrap=0,
)

stage_summary = DataFrame(
    stage=[s.name for s in staged.stages],
    status=[s.status for s in staged.stages],
    n_conditions=[s.n_conditions for s in staged.stages],
    selected_model=[isnothing(s.selected_model) ? "" : String(s.selected_model) for s in staged.stages],
    candidate_models=[join(s.candidate_models, "; ") for s in staged.stages],
    output_dir=[isnothing(s.output_dir) ? "" : String(s.output_dir) for s in staged.stages],
)

parameter_rows = NamedTuple[]
for (stage_name, params) in staged.parameter_bank
    for (param, value) in params
        push!(parameter_rows, (stage=stage_name, parameter=String(param), value=value))
    end
end
parameter_bank = DataFrame(parameter_rows)

CSV.write(joinpath(output_dir, "staged_stage_summary.csv"), stage_summary)
CSV.write(joinpath(output_dir, "staged_parameter_bank.csv"), parameter_bank)
CSV.write(joinpath(output_dir, "staged_failure_report.csv"), staged.failures)

println("Completed staged pipeline: ", staged.completed)
println("Stage summary: ", join(stage_summary.stage .* "=" .* stage_summary.status, ", "))
display(stage_summary)
display(parameter_bank)
if nrow(staged.failures) > 0
    display(staged.failures)
end

## Digestible Figures

This cell exports PNG plots that summarize the manufactured data, exposure profiles, fits, model comparisons, and staged pipeline results.


In [ ]:
using CSV
using DataFrames
using Plots

figure_dir = joinpath(output_dir, "figures")
mkpath(figure_dir)

default(size=(1200, 720), dpi=150, linewidth=2.5, markersize=4,
        guidefontsize=11, tickfontsize=9, legendfontsize=8)

manufactured = CSV.read(joinpath(output_dir, "manufactured_multistage_data.csv"), DataFrame)
manufactured.condition = manufactured.culture_type .* " | " .* manufactured.population_type .* " | dose=" .* string.(manufactured.dose)
p_data = plot(title="Manufactured multistage data", xlabel="Time", ylabel="Cell count", legend=:outerright)
for g in groupby(manufactured, :condition)
    plot!(p_data, g.time, g.count; marker=:circle, label=first(g.condition))
end
savefig(p_data, joinpath(figure_dir, "manufactured_multistage_counts.png"))

model_story = CSV.read(joinpath(output_dir, "manufactured_model_story.csv"), DataFrame)
p_story = bar(model_story.model_generator, model_story.rows; group=model_story.culture_type,
              xlabel="Generator model", ylabel="Rows", title="Concrete models used to manufacture the tour data",
              legend=:outerright, xrotation=35)
savefig(p_story, joinpath(figure_dir, "manufactured_model_generators.png"))

exposures = CSV.read(joinpath(output_dir, "exposure_profiles.csv"), DataFrame)
p_exposure = plot(exposures.time, exposures.constant; label="constant", xlabel="Time", ylabel="Exposure",
                  title="Exposure profiles", legend=:topright)
plot!(p_exposure, exposures.time, exposures.pulse; label="pulse")
plot!(p_exposure, exposures.time, exposures.stepped; label="stepped")
plot!(p_exposure, exposures.time, exposures.decaying; label="decaying")
savefig(p_exposure, joinpath(figure_dir, "exposure_profiles.png"))

model_examples = CSV.read(joinpath(output_dir, "model_examples.csv"), DataFrame)
p_models = bar(model_examples.model, model_examples.derivative_at_200; legend=false,
               xlabel="Model example", ylabel="Derivative at N=200",
               title="Composable model behavior", xrotation=35)
savefig(p_models, joinpath(figure_dir, "model_derivatives.png"))

single_compare = CSV.read(joinpath(output_dir, "single_compare_models.csv"), DataFrame)
p_bic = bar(single_compare.Model, single_compare.BIC; legend=false,
            xlabel="Model", ylabel="BIC", title="Single-condition model comparison")
savefig(p_bic, joinpath(figure_dir, "single_condition_bic.png"))

predictions = CSV.read(joinpath(output_dir, "single_compare_models_dict_predictions.csv"), DataFrame)
mono_naive = subset(manufactured, :culture_type => ByRow(==("monoculture")),
                   :population_type => ByRow(==("naive")), :dose => ByRow(==(0.0)))
p_fit = scatter(mono_naive.time, mono_naive.count; label="Observed", xlabel="Time",
                ylabel="Cell count", title="Single-condition fit overlay", legend=:topleft)
for g in groupby(predictions, :Model)
    plot!(p_fit, g.Time, g.Prediction; label=first(g.Model))
end
savefig(p_fit, joinpath(figure_dir, "single_condition_fit_overlay.png"))

stage_summary = CSV.read(joinpath(output_dir, "staged_stage_summary.csv"), DataFrame)
p_stage = bar(1:nrow(stage_summary), stage_summary.n_conditions; legend=false,
              xlabel="Pipeline stage", ylabel="Conditions", title="Completed staged pipeline",
              xticks=(1:nrow(stage_summary), replace.(stage_summary.stage, "_" => " ")),
              xrotation=35)
savefig(p_stage, joinpath(figure_dir, "staged_pipeline_conditions.png"))

params = CSV.read(joinpath(output_dir, "staged_parameter_bank.csv"), DataFrame)
p_param = bar(params.parameter, params.value; group=params.stage, xlabel="Parameter",
              ylabel="Value", title="Staged fitted parameter bank", legend=:outerright,
              yscale=:log10, xrotation=35)
savefig(p_param, joinpath(figure_dir, "staged_parameter_bank.png"))

figure_manifest = DataFrame(path=sort(relpath.(filter(isfile, readdir(figure_dir; join=true)), output_dir)))
CSV.write(joinpath(output_dir, "figure_manifest.csv"), figure_manifest)

produced_paths = String[]
for (root, _, files) in walkdir(output_dir)
    append!(produced_paths, [joinpath(root, file) for file in files])
end
produced_files = DataFrame(path=sort(relpath.(produced_paths, output_dir)))
CSV.write(joinpath(output_dir, "output_manifest.csv"), produced_files)

println("Generated figures: ", nrow(figure_manifest))
println("Figure directory: ", normpath(figure_dir))


## Output Manifest

This final cell lists the files produced by the function tour.

In [ ]:
produced_paths = String[]
for (root, _, files) in walkdir(output_dir)
    append!(produced_paths, [joinpath(root, file) for file in files])
end
produced_files = DataFrame(path=sort(relpath.(produced_paths, output_dir)))
CSV.write(joinpath(output_dir, "output_manifest.csv"), produced_files)
println("Function tour outputs written to: ", output_dir)
display(produced_files)